In [1]:
import pandas as pd
from sklearn.preprocessing import FunctionTransformer
%matplotlib inline

In [2]:
df_train = pd.read_csv('../data/raw/train.csv')
df_test = pd.read_csv('../data/raw/test.csv')

In [3]:
#Busque y eliminacion de features con mas del 85% de datos faltantes
initial_cols = df_train.columns
threshold = len(df_train) * 0.85
df_train = df_train.dropna(thresh=threshold, axis=1)
removed_cols = list(set(initial_cols) - set(df_train.columns))
print('columns removed --> ', len(removed_cols))
print(removed_cols)
#['PoolQC', 'Fence', 'Alley', 'LotFrontage', 'MiscFeature', 'MasVnrType', 'FireplaceQu']
#Estas columnas sera eliminadas en produccion antes de entrenar el modelo
#Y ademas antes de predecir

columns removed -->  7
['PoolQC', 'LotFrontage', 'MiscFeature', 'FireplaceQu', 'Fence', 'MasVnrType', 'Alley']


In [4]:
#Functions transaformers for imputing basement features
def impute_bsmt_features(df):
    bsmt_features = ['BsmtCond', 'BsmtQual', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']
    condition_not_basement = (df['TotalBsmtSF'] == 0) & (df['BsmtUnfSF'] == 0)
    df.loc[condition_not_basement, bsmt_features] = df.loc[condition_not_basement, bsmt_features].fillna('NA')
    return df

def impute_basement_exposure(df):
    df['BsmtExposure'] = df['BsmtExposure'].fillna(df.groupby(['BsmtCond', 'BsmtFinType1', 'BsmtFinType2', 'BsmtQual', 'BsmtFullBath', 'BsmtHalfBath'])['BsmtExposure']
                                                    .transform(lambda x: x.mode()[0] if not x.mode().empty else 'No'))
    return df

def impute_basement_type2(df):
    df['BsmtFinType2'] = df['BsmtFinType2'].fillna(df.groupby(['BsmtCond', 'BsmtQual', 'BsmtFullBath', 'BsmtHalfBath', 'BsmtExposure'])['BsmtFinType2']
                                                    .transform(lambda x: x.mode()[0] if not x.mode().empty else 'Unf'))
    return df

In [5]:
#Integeracion de funciones personalizadas (features engineering)
#1. Imputacion variables categoricas relacionadas con sótano

feature_engineering_steps = [
    ('impute_bsmt_features', FunctionTransformer(impute_bsmt_features, validate=False)),
    ('impute_basement_exposure', FunctionTransformer(impute_basement_exposure, validate=False)),
    ('impute_basement_type2', FunctionTransformer(impute_basement_type2, validate=False))
]